[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/drigaa/rEVA_CE4IB/HEAD?urlpath=%2Fdoc%2Ftree%2Fcasestudy.ipynb)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

import dm4bem as db

### Thermal Network

![thermal-network](images/Screenshot%202026-10-29%20093356.png)

### Material properties and geometry

In [ ]:
number_of_nodes = 10
number_of_branches = 14

In [ ]:
air = {
    'Density': 1.2,
    'Specific heat': 1000
}

# convection coefficients, W/(m²·K)
h = pd.DataFrame([{'in': 4., 'out': 10.}], index=['h'])

In [ ]:
l = 3

concrete = {
    'Conductivity': 1.400,
    'Density': 2300.0,
    'Specific heat': 880,
    'Width': 0.15,
    'Surface': l**2
}

insulation = {
    'Conductivity': 0.040,
    'Density': 16.0,
    'Specific heat': 1210,
    'Width': 0.25,
    'Surface': l**2
}

glass = {
    'U-Value': 0.6,
    'Surface': l**2
}

wall = pd.DataFrame.from_dict(
    {
        'Layer_out': insulation,
        'Layer_in': concrete,
        'window': glass
    },
    orient='index')

wall

### System of algebraic-differential equations DAE
#### Temp nodes and flow branches

In [ ]:
θ = [f'θ{i}' for i in range(number_of_nodes)]
q = [f'q{i}' for i in range(number_of_branches)]

#### Incidence matrix $A$


In [ ]:
A = np.zeros((number_of_branches, number_of_nodes))

A[0, 0] = 1
A[1, 0], A[1, 1] = -1, 1
A[2, 1], A[2, 2] = -1, 1
A[3, 2], A[3, 3] = -1, 1
A[4, 3], A[4, 4] = -1, 1
A[5, 4], A[5, 5] = -1, 1
A[6, 6], A[6, 5] = -1, 1
A[7, 7], A[7, 6] = -1, 1
A[8, 8], A[8, 7] = -1, 1
A[9, 9], A[9, 8] = -1, 1
A[10, 9] = 1
A[11, 5] = 1
A[12, 9] = 1
A[13, 5] = 1

A = pd.DataFrame(A, index=q, columns=θ)

#### Conductance Matrix $G$

In [ ]:
## P-controler gain
Kp = 0
G = np.zeros(number_of_branches)

G[[1, 2]] = (wall["Conductivity"] / (wall["Width"] / 2) * wall["Surface"]).Layer_out
G[[3, 4, 7 ,8]] =  (wall["Conductivity"] / (wall["Width"] / 2) * wall["Surface"]).Layer_in

G[[0]] = h['out'] * wall["Surface"].Layer_out
G[[5, 6, 9]] = h['in'] * wall["Surface"].Layer_in

G[[12, 13]] = (wall['U-Value'] * wall["Surface"]).window

G[[10, 11]] = Kp
G = pd.Series(G, index=q)
G

#### Capacity Matrix $C$

In [ ]:
C = np.zeros(number_of_nodes)

C[1] = (wall['Width'] * wall['Surface'] * wall['Density'] * wall["Specific heat"]).Layer_out
C[[3, 7]] = (wall['Width'] * wall['Surface'] * wall['Density'] * wall["Specific heat"]).Layer_in
C[[5, 9]] = air["Density"] * l**3 * air["Specific heat"]
C = pd.Series(C, index=θ)
C

#### Temperatur source vector $b$

In [ ]:
b = pd.Series(np.zeros(number_of_branches), index=q)
b[[0, 12, 13]] = "T_0"
b[[10, 11]] = "Ti_sp"
b

#### Heat flow source vector $f$

In [ ]:
f = pd.Series(np.zeros(number_of_nodes), index=θ)
f[[0]] = 'Φo'
f[[5, 9]] = 'Qa'
f

#### Output vector $y$

In [ ]:
y = pd.Series(np.zeros(number_of_nodes), index=θ)
y[[5, 9]] = 1
y

In [ ]:
TC = {"A": A,
      "G": G,
      "C": C,
      "b": b,
      "f": f,
      "y": y}


In [ ]:
[As, Bs, Cs, Ds, us] = db.tc2ss(TC)